# AgriVision: MODIS + Sentinel-2 fusion pipeline (synthetic sample data)
Follows page 3 of `docs/diagrams/AgriVision_architecture.drawio`, one section per box.
Module 1 simulates the GEE exports, so Modules 2–9 are the real pipeline and run unchanged on real data.
Colab: *Runtime → Change runtime type → GPU*, then *Run all*. Local RTX 50-series: PyTorch built for CUDA 12.8+.

## 0 · Setup

In [ ]:
# installs the two packages Colab lacks
%pip install -q rasterio pytorch-msssim

In [ ]:
import os, datetime as dt                                              # folders, calendar dates
import numpy as np, pandas as pd, matplotlib.pyplot as plt             # arrays, tables, plots
from scipy.ndimage import gaussian_filter, map_coordinates, binary_dilation, shift as nd_shift  # image operations
from scipy.signal import savgol_filter                                 # time-series smoothing (M9)
from skimage.registration import phase_cross_correlation               # image matching (2.3)
from skimage.metrics import structural_similarity                      # SSIM score (M8)
import rasterio                                                        # GeoTIFF read/write
from rasterio.transform import from_origin                             # builds a grid's georeference
from rasterio.warp import reproject, Resampling                        # moves images between grids
import torch, torch.nn as nn, torch.nn.functional as F                 # neural network
from torch.utils.data import Dataset, DataLoader                       # batching training samples
from pytorch_msssim import ms_ssim                                     # MS-SSIM loss term
from IPython.display import display                                    # show tables

rng = np.random.default_rng(42); torch.manual_seed(42)                 # identical synthetic scene every run
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'                   # use the GPU when there is one
print('device:', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU')  # confirm which one
try:                                                                   # in Colab, keep outputs on Google Drive
    # mount Drive and save there
    from google.colab import drive; drive.mount('/content/drive'); ROOT = '/content/drive/MyDrive/agrivision'
except ImportError:                                                    # anywhere else, a local folder
    ROOT = 'agrivision'  # local output folder
os.makedirs(f'{ROOT}/predicted', exist_ok=True)                        # where Module 7 writes GeoTIFFs

H = W = 750                                                            # scene: 750 × 750 S2 pixels = 7.5 km
RES_F, RES_C, K = 10.0, 250.0, 25                                      # 10 m fine grid, 250 m common grid, 25 px per cell
HC = WC = H // K                                                       # common grid is 30 × 30 cells
MOD_RES = 231.656358                                                   # true size of a MODIS "250 m" cell
X0, Y0, CRS = 500000.0, 3500000.0, 'EPSG:32643'                        # scene's upper-left corner, UTM zone 43N
FINE_TF = from_origin(X0, Y0, RES_F, RES_F)                            # georeference of the 10 m grid
COMMON_TF = from_origin(X0, Y0, RES_C, RES_C)                          # georeference of the 250 m grid
START, N_DAYS = dt.date(2023, 11, 1), 150                              # season start and length (days)
date = lambda d: START + dt.timedelta(days=int(d))                     # day number → calendar date

---
## Module 1 · Data (GEE) — synthetic
Builds a farm scene and simulates the three products GEE would deliver: **Sentinel-2 L2A** (B4, B8, Cloud Score+),
**MOD09GQ** (red, NIR) and **MOD09GA** (cloud flags, view angle), with their real problems: clouds and shadows,
S2 misregistration, MODIS's 231.66 m grid, blur that grows off-nadir, ~50 m geolocation error, band differences.
Three wheat fields get crop stress partway through the season (the answer key for Module 9).

In [ ]:
yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)                         # row/column of every pixel
seeds = rng.uniform(0, H, (60, 2))                                     # 60 random field centres
field_id = np.argmin([(yy - y) ** 2 + (xx - x) ** 2 for y, x in seeds], axis=0)  # pixel belongs to nearest centre
N_FIELDS = len(seeds)                                                  # number of fields
crop = rng.choice(['wheat', 'mustard', 'fallow', 'orchard', 'water'], N_FIELDS, p=[.45, .22, .13, .12, .08])  # cover type
road = (np.abs(yy - 311) <= 1) | (np.abs(xx - 456) <= 1)               # two 3-pixel-wide roads
field_id[road] = -1                                                    # roads belong to no field

def growth(t, fmax, sow, harvest, r=5.5):                              # green cover over the season (rise, plateau, fall)
    # bare-soil level + crop cover between sowing and harvest curves
    return 0.04 + (fmax - 0.04) * (1 / (1 + np.exp(-(t - sow) / r)) - 1 / (1 + np.exp(-(t - harvest) / r)))

t = np.arange(N_DAYS)                                                  # day numbers
cover = np.zeros((N_FIELDS, N_DAYS), np.float32)                       # green cover per field per day
for i, c in enumerate(crop):                                           # each field gets its own dates and vigour
    # wheat: vigour, sowing day, harvest day
    if c == 'wheat': cover[i] = growth(t, rng.uniform(.82, .95), rng.uniform(30, 42), rng.uniform(118, 132))
    # mustard: earlier and shorter season
    if c == 'mustard': cover[i] = growth(t, rng.uniform(.70, .85), rng.uniform(15, 25), rng.uniform(88, 100))
    if c == 'orchard': cover[i] = rng.uniform(.5, .6)                  # evergreen trees
    if c == 'fallow': cover[i] = 0.05                                  # bare soil
soil = rng.uniform(.17, .24, N_FIELDS)                                 # soil brightness (red) per field
patchy = np.clip(1 + 4 * gaussian_filter(rng.standard_normal((H, W)), 12), .8, 1.15)  # uneven growth inside fields

# pick 3 wheat fields and a stress start day (70-95) for each
STRESS = [dict(field=f, start=int(rng.uniform(70, 95))) for f in rng.choice(np.flatnonzero(crop == 'wheat'), 3, replace=False)]
for s in STRESS:                                                       # stress hits a disc inside each chosen field
    cy, cx = np.argwhere(field_id == s['field']).mean(0)               # field centre
    # disc of random radius, clipped to the field
    s['mask'] = ((yy - cy) ** 2 + (xx - cx) ** 2 < rng.uniform(35, 60) ** 2) & (field_id == s['field'])

def true_reflectance(d):                                               # noise-free 10 m (red, NIR) on day d; simulator only
    f = np.clip(field_id, 0, None)                                     # roads temporarily read field 0's values
    fv = cover[f, d] * patchy                                          # green cover of every pixel
    for s in STRESS:                                                   # stressed discs lose up to 40 % cover over 12 days
        # cover shrinks inside the disc after the start day
        fv = np.where(s['mask'], fv * (1 - .4 * np.clip((d - s['start']) / 12, 0, 1)), fv)
    fv = np.clip(fv, 0, 1)                                             # cover is a fraction
    red = fv * .03 + (1 - fv) * soil[f]                                # vegetation is dark in red, soil bright
    nir = fv * .48 + (1 - fv) * (1.3 * soil[f] + .02)                  # vegetation is bright in NIR
    red, nir = np.where(crop[f] == 'water', .04, red), np.where(crop[f] == 'water', .02, nir)  # water is dark
    red, nir = np.where(road, .16, red), np.where(road, .20, nir)      # roads are bright and constant
    return np.stack([red, nir]).astype(np.float32)  # 2 × H × W reflectance

def clouds(frac):                                                      # smooth cloud blobs covering `frac` of the scene
    n = gaussian_filter(rng.standard_normal((H, W)), 28)               # smooth random field
    # top `frac` of the field = cloud (none if frac = 0)
    return n > np.quantile(n, 1 - frac) if frac > 0 else np.zeros((H, W), bool)

cloud_frac = lambda: rng.choice([0, rng.uniform(.05, .3), rng.uniform(.4, .95)], p=[.55, .15, .30])  # clear/partly/mostly
with_shadow = lambda cl: (cl, np.roll(cl, (18, 22), (0, 1)) & ~cl)    # shadow = cloud mask displaced by the sun angle

# --- Sentinel-2 L2A: every 5 days, ×10000 integers like GEE ---
S2_DAYS = list(range(2, N_DAYS, 5))                                    # acquisition days
s2_raw, sky = {}, {}                                                   # raw S2 products; cloud/shadow masks per day
for d in S2_DAYS:  # every Sentinel-2 date
    sky[d] = cl, sh = with_shadow(clouds(cloud_frac()))                # this day's clouds and shadows
    r = nd_shift(true_reflectance(d), (0, *rng.uniform(-1.5, 1.5, 2)), order=1, mode='nearest')  # sub-pixel misregistration
    r = np.where(sh, r * .45, r + rng.normal(0, .004, r.shape))        # shadows darken; sensor noise elsewhere
    r = np.where(cl, np.array([.38, .42])[:, None, None], r)           # clouds are bright in both bands
    cs = np.clip(.95 - .9 * gaussian_filter(cl * 1., 2) - .6 * sh + rng.normal(0, .03, (H, W)), 0, 1)  # Cloud Score+ cs_cdf
    # store like GEE: B4, B8 as reflectance × 10000, plus cs_cdf
    s2_raw[d] = dict(B4=(r[0] * 1e4).clip(0, 1e4).astype(np.uint16), B8=(r[1] * 1e4).clip(0, 1e4).astype(np.uint16), cs_cdf=cs)

# --- MODIS MOD09GQ (daily red/NIR on its own 231.66 m grid) + MOD09GA (1 km cloud bits, view angle) ---
MOD_X0, MOD_Y0 = X0 - 737.3, Y0 + 761.7                                # MODIS grid origin: offset from the scene, with margin
MOD_N = int(np.ceil((W * RES_F + 1400) / MOD_RES))                     # cells per side (covers scene + margin)
MOD_TF = from_origin(MOD_X0, MOD_Y0, MOD_RES, MOD_RES)                 # georeference of the MODIS grid
ctr = (np.arange(MOD_N) + .5) * MOD_RES                                # cell centres, metres from the MODIS origin
MR, MC = np.meshgrid((Y0 - MOD_Y0 + ctr) / RES_F - .5, (MOD_X0 + ctr - X0) / RES_F - .5, indexing='ij')  # centres in fine px
GAIN, OFFSET = np.array([.94, .92])[:, None, None], np.array([.008, .015])[:, None, None]  # MODIS vs S2 band difference
to_1km = lambda a: np.pad(a, (0, -MOD_N % 4), mode='edge').reshape(-1, 4, -(-MOD_N // 4), 4).mean((1, 3))  # 4×4 cells → 1 km
mod09gq, mod09ga = {}, {}                                              # daily products
for d in range(N_DAYS):  # every day of the season
    vz = rng.uniform(0, 55)                                            # sensor view angle today (degrees)
    sigma = np.hypot(MOD_RES / RES_F / 12 ** .5, 7 * (1 + 1.5 * (vz / 55) ** 2))  # cell area + optics blur, wider off-nadir
    err = np.array([3., -2.]) + rng.normal(0, 1.5, 2)                  # position error: systematic + daily (fine px)
    cl, sh = sky[d] if d in sky else with_shadow(clouds(cloud_frac())) # same sky as S2 on S2 days (both pass ~10:30)
    # what one MODIS cell records: blurred scene sampled at its (misplaced) centre
    see = lambda img: map_coordinates(gaussian_filter(img, sigma, mode='nearest'), [MR + err[0], MC + err[1]], order=1, mode='nearest')
    cf, sf = see(cl * 1.), see(sh * 1.)                                # cloud and shadow fraction inside each cell
    m = GAIN * np.stack([see(b) for b in true_reflectance(d)]) + OFFSET  # blurred, misplaced, spectrally different
    m = m * (1 - .55 * sf) * (1 - cf) + np.array([.38, .42])[:, None, None] * cf + rng.normal(0, .006, m.shape)  # clouds + noise
    # store like GEE: reflectance × 10000
    mod09gq[d] = dict(sur_refl_b01=np.round(m[0] * 1e4).astype(np.int16), sur_refl_b02=np.round(m[1] * 1e4).astype(np.int16))
    c1, s1 = to_1km(cf), to_1km(sf)                                    # cloud/shadow fraction per 1 km cell
    state = np.where(c1 > .6, 1, np.where(c1 > .02, 2, 0))             # bits 0-1: 0 clear, 1 cloudy, 2 mixed
    state |= (s1 > .3).astype(int) << 2 | (binary_dilation(c1 > .02) & (c1 <= .02)).astype(int) << 13  # bit 2 shadow, bit 13 adjacent
    mod09ga[d] = dict(state_1km=state.astype(np.uint16), SensorZenith=np.full(c1.shape, int(vz * 100), np.int16))  # angle ×100
print(len(S2_DAYS), 'S2 images,', N_DAYS, 'MODIS days')  # summary

---
## Module 2 · Preprocessing (reflectance kept, no NDVI)
### Sentinel-2 branch
**2.1 Cloud mask:** DN → reflectance; drop pixels with Cloud Score+ `cs_cdf < 0.65`, plus a 3-pixel buffer
around them (cloud edges are under-flagged).

In [ ]:
def s2_mask(raw):                                                      # one S2 image → masked reflectance
    r = np.stack([raw['B4'], raw['B8']]) / 1e4                         # GEE scale factor
    r[:, binary_dilation(raw['cs_cdf'] < .65, iterations=3)] = np.nan  # clouds, shadows and their edges → invalid
    return r.astype(np.float32)  # masked reflectance

s2 = {d: s2_mask(s2_raw[d]) for d in S2_DAYS}                          # all dates
clear = {d: np.isfinite(s2[d][0]).mean() for d in S2_DAYS}             # clear fraction per date
print({d: round(float(c), 2) for d, c in clear.items()})  # clear fraction per S2 date

**2.2 Common UTM grid (10 m):** reprojects an image onto the shared grid (bilinear). The same function is used
for MODIS in 2.5. Our synthetic S2 is already on the grid, so for S2 this is a pass-through.

In [ ]:
def to_grid(img, src_tf, dst_tf, shape, src_crs=CRS):                  # bilinear reprojection; NaN stays NaN
    out = np.full((img.shape[0], *shape), np.nan, np.float32)          # empty target grid
    # GDAL bilinear reprojection from the source grid to the target grid
    reproject(np.ascontiguousarray(img, np.float32), out, src_transform=src_tf, src_crs=src_crs, dst_transform=dst_tf,
              # ...bilinear; NaN marks no data on both sides
              dst_crs=CRS, resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return out  # the result

s2 = {d: to_grid(s2[d], FINE_TF, FINE_TF, (H, W)) for d in S2_DAYS}    # S2 onto the 10 m grid

**2.3 Co-registration:** a 1-pixel shift between S2 dates looks like change at every field edge. Match each date's
NIR to the clearest date (phase correlation) and shift it back. Real data: AROSICS `COREG_LOCAL` does this per region.

In [ ]:
fill = lambda a: np.where(np.isfinite(a), a, np.nanmean(a))            # gaps → mean (matching needs complete images)

def shift_img(img, s):                                                 # shift by s pixels; gaps move with the image
    out = np.stack([nd_shift(np.nan_to_num(b), s, order=1, mode='nearest') for b in img])  # shift values
    out[:, nd_shift(np.isfinite(img[0]) * 1., s, order=1) < .999] = np.nan  # shift the valid mask too
    return out  # the result

MASTER = max(S2_DAYS, key=clear.get)                                   # clearest date is the reference
for d in S2_DAYS:  # every Sentinel-2 date
    if d != MASTER and clear[d] >= .3:                                 # under 30 % clear: too little to match
        # measure the shift vs the master date on NIR, then undo it on both bands
        s2[d] = shift_img(s2[d], phase_cross_correlation(fill(s2[MASTER][1]), fill(s2[d][1]), upsample_factor=20)[0])

### MODIS branch
**2.4 Cloud + view-angle mask:** MOD09GQ has no cloud flags, so take them from MOD09GA (same day): keep cells that
are clear (bits 0-1 = 0), shadow-free (bit 2) and not next to cloud (bit 13), seen within 40° of nadir.

In [ ]:
def modis_mask(gq, ga):                                                # one MODIS day → masked reflectance
    up = lambda a: np.kron(a, np.ones((4, 4), a.dtype))[:MOD_N, :MOD_N]  # 1 km layer onto the 231 m cells
    st, zen = up(ga['state_1km']), up(ga['SensorZenith']) / 100        # flags and view angle per cell
    ok = ((st & 3) == 0) & ((st & 1 << 2) == 0) & ((st & 1 << 13) == 0) & (zen < 40)  # clear, no shadow, not adjacent, near nadir
    r = np.stack([gq['sur_refl_b01'], gq['sur_refl_b02']]) / 1e4       # GEE scale factor
    r[:, ~ok] = np.nan                                                 # drop bad cells
    return r.astype(np.float32)  # masked reflectance

modis = {d: modis_mask(mod09gq[d], mod09ga[d]) for d in range(N_DAYS)} # all days
modis_ok = {d: np.isfinite(modis[d][0, 4:-4, 4:-4]).mean() for d in range(N_DAYS)}  # valid fraction over the scene
# how many MODIS days are almost fully usable
print(sum(v >= .95 for v in modis_ok.values()), 'of', N_DAYS, 'MODIS days are ≥ 95 % usable')

**2.5 Bilinear resample:** MODIS (231.66 m, own origin) onto the 250 m common grid (to compare with S2 in 2.6-2.7)
and onto the 10 m grid (what the network takes). Real MODIS: pass its sinusoidal CRS as `src_crs`.

In [ ]:
modis_250 = {d: to_grid(modis[d], MOD_TF, COMMON_TF, (HC, WC)) for d in range(N_DAYS)}  # 30 × 30 per day
modis_10_raw = lambda d: to_grid(modis[d], MOD_TF, FINE_TF, (H, W))    # 750 × 750, made on demand (saves memory)

**2.6 PSF + offset fit:** a MODIS cell is blurrier than its size and slightly misplaced. On days both sensors are
clear, find the blur (sigma) and offset (dy, dx) that make S2, blurred and averaged to 250 m, best match MODIS.
Other days use the median offset of the 3 nearest fitted days. MODIS is then shifted back by (−dy, −dx).

In [ ]:
def fit_psf_offset(fine, coarse):                                      # search blur × offset for the best correlation
    best, ok = (-1, 0, 0, 0), np.isfinite(coarse)                      # (r, sigma, dy, dx); usable MODIS cells
    for s in (0, 4, 8, 12, 16, 20, 24):                                # candidate blur widths (fine px)
        g = np.pad(gaussian_filter(fill(fine), s), 20, mode='edge')    # blurred S2, padded so it can be shifted
        # shift blurred S2 by (dy, dx), average to 250 m, correlate with MODIS
        score = lambda dy, dx: np.corrcoef(g[20 - dy:20 - dy + H, 20 - dx:20 - dx + W].reshape(HC, K, WC, K).mean((1, 3))[ok], coarse[ok])[0, 1]
        _, dy, dx = max((score(a, b), a, b) for a in range(-20, 21, 4) for b in range(-20, 21, 4))  # coarse search, 4 px steps
        # refine ±3 px around the coarse best and keep the overall best
        best = max(best, max((score(a, b), s, a, b) for a in range(dy - 3, dy + 4) for b in range(dx - 3, dx + 4)
                             if abs(a) <= 20 and abs(b) <= 20))        # refine around it, 1 px steps
    return dict(zip(('r', 'sigma', 'dy', 'dx'), best))  # best correlation, blur and offset

PAIRS = [d for d in S2_DAYS if clear[d] >= .95 and modis_ok[d] >= .95] # days both sensors are clear
fit = {d: fit_psf_offset(s2[d][1], modis_250[d][1]) for d in PAIRS}    # fit on NIR (strongest field contrast)
display(pd.DataFrame(fit).T.round(3))  # table of fitted blur/offset per clear day

def offset(d):                                                         # (dy, dx) to undo for any day
    near = sorted(fit, key=lambda p: abs(p - d))[:1 if d in fit else 3]  # its own fit, or 3 nearest fitted days
    return np.median([[fit[p]['dy'], fit[p]['dx']] for p in near], axis=0)  # median offset of those days

**2.7 Band adjustment:** MODIS and S2 bands see slightly different wavelengths. Regress S2 (degraded with each day's
fitted blur and offset) on MODIS: `S2 = a·MODIS + b` per band, then apply to all MODIS.

In [ ]:
def degrade(fine, f):                                                  # S2 as MODIS would see it
    # blur with the fitted sigma and shift by the fitted offset
    g = np.stack([nd_shift(gaussian_filter(fill(b), f['sigma']), (f['dy'], f['dx']), order=1, mode='nearest') for b in fine])
    return g.reshape(2, HC, K, WC, K).mean((2, 4))                     # average to 250 m

BAND_ADJ = []                                                          # (a, b) per band
for b in range(2):  # red, then NIR
    x = np.concatenate([modis_250[d][b].ravel() for d in fit])         # MODIS values on all clear pairs
    y = np.concatenate([degrade(s2[d], fit[d])[b].ravel() for d in fit])  # matching degraded S2 values
    ok = np.isfinite(x) & np.isfinite(y)                               # usable cells
    BAND_ADJ.append(np.polyfit(x[ok], y[ok], 1))                       # straight-line fit
print('band adjustment (a, b): red', BAND_ADJ[0].round(4), '| nir', BAND_ADJ[1].round(4))  # fitted coefficients

**Output of Module 2:** `s2[d]` (masked, co-registered) and `modis10(d)` (resampled → shifted back → band-adjusted),
both red + NIR reflectance on the 10 m grid. No NDVI.

In [ ]:
_cache = {}                                                            # finished MODIS days, computed once
def modis10(d):                                                        # final 10 m MODIS for day d
    if d not in _cache:  # compute only the first time
        m = shift_img(modis_10_raw(d), -offset(d))                     # undo the position error
        _cache[d] = np.stack([a * m[i] + c for i, (a, c) in enumerate(BAND_ADJ)]).astype(np.float32)  # band adjustment
    return _cache[d]  # cached result

---
## Module 3 · Dataset
Samples **(C1, F1, C2) → F2** from pairs of clear dates ≤ 32 days apart. Dates (not patches) are split into
train / validation / test, so test dates are never seen. 256 px patches, stride 128, skipped if any image is > 5 %
invalid; random flips/rotations for training.

In [ ]:
TEST = PAIRS[1:-1][1::3]                                               # held-out dates (never first/last: need neighbours)
VAL = [d for d in PAIRS[1:-1][::3] if d not in TEST][:2]               # validation dates
TRAIN = [d for d in PAIRS if d not in TEST + VAL]                      # training dates
print('train', TRAIN, '| val', VAL, '| test', TEST)  # the date split

class Patches(Dataset):  # training samples: (C1, F1, C2) → F2 patches
    def __init__(self, targets, refs, augment):                        # every (reference, target) pair × patch position
        self.augment = augment                                         # flips/rotations on or off
        stack = lambda t1, t2: (modis10(t1), s2[t1], modis10(t2), s2[t2])  # C1, F1, C2, F2
        # every reference/target pair ≤ 32 days apart...
        self.items = [(t1, t2, i, j) for t2 in targets for t1 in refs if 0 < abs(t2 - t1) <= 32
                      # ...every 256 px patch position, stride 128...
                      for i in range(0, H - 255, 128) for j in range(0, W - 255, 128)
                      # ...kept only if all four images are ≤ 5 % invalid
                      if max(np.isnan(a[0, i:i + 256, j:j + 256]).mean() for a in stack(t1, t2)) <= .05]
        self.stack = stack                                             # keep for __getitem__

    def __len__(self):  # dataset size
        return len(self.items)                                         # number of samples

    def __getitem__(self, k):  # one sample
        t1, t2, i, j = self.items[k]                                   # which pair and where
        c1, f1, c2, f2 = (a[:, i:i + 256, j:j + 256] for a in self.stack(t1, t2))  # cut the patch
        x = np.stack([c1, np.where(np.isnan(f1), c1, f1), c2, np.where(np.isnan(f2), c2, f2)])  # rare gaps → MODIS value
        x = torch.from_numpy(np.nan_to_num(x)).clamp(0, 1)             # to tensor
        if self.augment:                                               # random flip + random 90° rotation
            # same flip/rotation for all four images
            x = torch.rot90(x.flip(-1) if torch.rand(1) < .5 else x, int(torch.randint(4, (1,))), (-2, -1))
        return tuple(x)                                                # c1, f1, c2, f2

# training batches of 8, shuffled, augmented
train_dl = DataLoader(Patches(TRAIN, TRAIN, True), batch_size=8, shuffle=True, drop_last=True)
val_dl = DataLoader(Patches(VAL, TRAIN, False), batch_size=8)  # validation batches: unseen target dates, no augmentation
print(len(train_dl.dataset), 'training patches,', len(val_dl.dataset), 'validation patches')  # dataset sizes

---
## Module 4 · Fusion network (reflectance in → reflectance out)
**Fine encoder** reads field detail from F1; **change encoder** learns the fine-scale change from [C1, F1, C2]
(last layer linear: change can be negative); **fusion** adds them; **decoder** turns features back into red + NIR.
Optional upgrades go here (second reference date, U-Net / dilated change encoder, GAN-STFM, SwinSTFM, STFDiff).

In [ ]:
conv = lambda i, o: nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(i, o, 3))  # 3×3 conv that keeps the image size

class FusionNet(nn.Module):  # the fusion network
    def __init__(self, b=2):                                           # b = number of bands (red, NIR)
        super().__init__()  # PyTorch module setup
        # fine encoder: F1 → 128 feature maps of field detail
        self.fine_enc = nn.Sequential(conv(b, 32), nn.ReLU(), conv(32, 64), nn.ReLU(), conv(64, 128), nn.ReLU())
        # change encoder: [C1, F1, C2] → 128 maps of change (last layer linear)
        self.change_enc = nn.Sequential(conv(3 * b, 32), nn.ReLU(), conv(32, 64), nn.ReLU(), conv(64, 128))
        # decoder: 128 maps → red, NIR
        self.decoder = nn.Sequential(conv(128, 64), nn.ReLU(), conv(64, 32), nn.ReLU(), nn.Conv2d(32, b, 1))

    def forward(self, c1, f1, c2):                                     # predicted F2 = decode(detail + change)
        # fusion = add detail and change, then decode
        return self.decoder(self.fine_enc(f1) + self.change_enc(torch.cat([c1, f1, c2], 1)))

model = FusionNet().to(DEV)                                            # build it on the GPU
print(sum(p.numel() for p in model.parameters()), 'parameters')  # model size

---
## Modules 5–6 · Loss & training
Loss on predicted red + NIR: L1 + 0.8·(1 − MS-SSIM) + 0.5·L1 on NDVI **computed from the predicted bands**
(a training signal only; the NDVI maps are made in Module 9). Adam, lr 1e-3, halved when validation stalls.

In [ ]:
ndvi_t = lambda x: (x[:, 1] - x[:, 0]) / (x[:, 1] + x[:, 0] + 1e-6)    # NDVI of a batch of predictions

def loss_fn(p, t):                                                     # pixel + structure + NDVI terms
    # L1 + 0.8·(1 − MS-SSIM) + 0.5·NDVI L1
    return F.l1_loss(p, t) + .8 * (1 - ms_ssim(p.clamp(0, 1), t.clamp(0, 1), data_range=1)) + .5 * F.l1_loss(ndvi_t(p), ndvi_t(t))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)                    # optimiser (Paper 1's setting)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=.5, patience=4)  # halve lr when stuck
EPOCHS, best = int(os.environ.get('EPOCHS', 40)), np.inf               # training length; best validation so far
for ep in range(EPOCHS):  # each epoch
    model.train()                                                      # training mode
    for c1, f1, c2, f2 in train_dl:  # each training batch
        opt.zero_grad()                                                # clear old gradients
        loss = loss_fn(model(c1.to(DEV), f1.to(DEV), c2.to(DEV)), f2.to(DEV))  # predict and score against real F2
        loss.backward(); opt.step()                                    # update weights
    model.eval()                                                       # evaluation mode
    with torch.no_grad():                                              # validation L1 on unseen dates
        # mean L1 over validation batches
        val = np.mean([F.l1_loss(model(c1.to(DEV), f1.to(DEV), c2.to(DEV)), f2.to(DEV)).item() for c1, f1, c2, f2 in val_dl])
    sched.step(val)                                                    # maybe lower the learning rate
    if val < best: best = val; torch.save(model.state_dict(), f'{ROOT}/best.pt')  # keep the best weights
    print(f'epoch {ep + 1}: train loss {loss.item():.4f} | val L1 {val:.4f}')  # progress
_ = model.load_state_dict(torch.load(f'{ROOT}/best.pt', map_location=DEV))  # use the best epoch from here on

---
## Module 7 · Inference
Every MODIS-only day: nearest clear S2 date as reference (F1, C1) + that day's MODIS (C2), predicted in 256 px tiles
with 32 px overlap (only tile centres kept, so no seams). Reference pool = train + validation dates (test dates
stay hidden for Module 8; add them back in production).

In [ ]:
POOL = sorted(TRAIN + VAL)                                             # dates with real S2 used as references
DAYS = [d for d in range(N_DAYS) if d not in POOL and modis_ok[d] >= .9]  # days to predict (usable MODIS)
nearest = lambda d, pool: min(pool, key=lambda r: (abs(r - d), r > d)) # closest reference date (earlier wins ties)
filled = lambda d: np.where(np.isnan(s2[d]), modis10(d), s2[d])        # S2 with rare gaps patched by MODIS

def predict(c1, f1, c2, tile=256, pad=32):                             # whole scene, tile by tile
    step, n = tile - 2 * pad, -(-H // (tile - 2 * pad)) * (tile - 2 * pad)  # kept part of a tile; padded scene size
    # C1, F1, C2 padded (reflect) and moved to the GPU
    x = [torch.from_numpy(np.pad(np.nan_to_num(a), ((0, 0), (pad, n - H + pad), (pad, n - W + pad)), mode='reflect'))[None].float().to(DEV)
         for a in (c1, f1, c2)]                                        # pad so every tile is full
    out = np.zeros((2, n, n), np.float32)                              # stitched result
    with torch.no_grad():  # no gradients needed
        for i in range(0, n, step):  # tile rows
            for j in range(0, n, step):                                # predict a tile, keep its centre
                # predict a 256 px tile, keep its 192 px centre
                out[:, i:i + step, j:j + step] = model(*[a[..., i:i + tile, j:j + tile] for a in x])[0, :, pad:-pad, pad:-pad].cpu().numpy()
    return out[:, :H, :W].clip(0, 1)                                   # crop the padding

def fuse(d, pool=POOL):                                                # fusion prediction for day d
    r = nearest(d, pool)                                               # reference date
    out = predict(modis10(r), filled(r), modis10(d))                   # C1, F1, C2 → F2
    out[:, np.isnan(modis10(d)[0])] = np.nan                           # no MODIS there → no prediction
    return out  # the result

**Optional · per-band blend (SOWCEM idea, on reflectance):** blend the model output with a **time guess** (linear
interpolation between the clear S2 dates around the day), per pixel and band, with weights `w_S + w_T = 1` that
minimise the combined squared error. Errors are measured on validation dates (unseen by the model) and smoothed
spatially so a few dates give stable weights.

In [ ]:
def time_guess(d, pool=POOL):                                          # linear interpolation in time, per pixel
    a = max([r for r in pool if r <= d], default=min(pool))            # clear date before (or first)
    b = min([r for r in pool if r >= d], default=max(pool))            # clear date after (or last)
    w = 0 if a == b else (d - a) / (b - a)                             # how far between them
    return (1 - w) * filled(a) + w * filled(b)  # weighted mix of the two clear images

E = np.zeros((3, 2, H, W))                                             # sums of eS², eT², eS·eT per pixel and band
for v in VAL:                                                          # hide each validation date and guess it
    pool = [r for r in POOL if r != v]                                 # references without it
    eS, eT = np.nan_to_num(s2[v] - time_guess(v, pool)), np.nan_to_num(s2[v] - fuse(v, pool))  # both errors
    E += [eS ** 2, eT ** 2, eS * eT]  # add this date's error products
eSS, eTT, eST = (np.stack([gaussian_filter(b, 5) for b in e]) for e in E)  # smooth the error statistics
W_S = np.clip((eTT - eST) / (eSS + eTT - 2 * eST + 1e-12), 0, 1).astype(np.float32)  # optimal weight on time guess
USE_BLEND = True                                                       # set False to output the model alone
# average weights per band
print('mean weight on time guess', W_S.mean((1, 2)).round(2), '| on model', (1 - W_S).mean((1, 2)).round(2))

**Predicted Sentinel-2-like image:** one GeoTIFF per predicted day (red + NIR, 10 m, ×10000 int16, EPSG:32643).

In [ ]:
model_only = {}                                                        # raw model output on test dates (for Module 8)
for d in DAYS:  # every day to predict
    out = fuse(d)                                                      # model prediction
    if d in TEST: model_only[d] = out                                  # keep for comparison
    final = np.where(np.isnan(out), np.nan, W_S * time_guess(d) + (1 - W_S) * out) if USE_BLEND else out  # blended image
    # new GeoTIFF: 2 bands, int16, 10 m UTM grid
    with rasterio.open(f'{ROOT}/predicted/{date(d)}.tif', 'w', driver='GTiff', height=H, width=W, count=2, dtype='int16',
                       # ...georeferenced, -9999 = no data, compressed
                       crs=CRS, transform=FINE_TF, nodata=-9999, compress='deflate') as f:
        f.write(np.where(np.isnan(final), -9999, np.round(final * 1e4)).astype(np.int16))  # write red, NIR

def read_pred(d):                                                      # GeoTIFF → reflectance (NaN = no data)
    with rasterio.open(f'{ROOT}/predicted/{date(d)}.tif') as f:  # open the day's GeoTIFF
        a = f.read().astype(np.float32)  # read both bands
    return np.where(a == -9999, np.nan, a / 1e4)  # back to reflectance
print(len(DAYS), 'GeoTIFFs written to', f'{ROOT}/predicted')  # summary

---
## Module 8 · Evaluation (held-out real S2 dates)
Predicted image vs real S2 on test dates: image metrics (RMSE, SSIM, SAM, ERGAS), then NDVI made from each.
Baselines: reuse F1, time interpolation, and **ESTARFM** (vectorised; similar-pixel candidates sampled every 5 px).

In [ ]:
def estarfm(F1, C1, F2, C2, Cp, win=51, step=5):                       # two reference pairs + coarse at prediction date
    T = lambda a: torch.tensor(np.nan_to_num(a), dtype=torch.float32, device=DEV)  # numpy → GPU tensor
    F1, C1, F2, C2, Cp = map(T, (F1, C1, F2, C2, Cp)); r = win // 2    # to GPU tensors; window radius
    Fs, Cs = torch.cat([F1, F2]), torch.cat([C1, C2])                  # fine / coarse stacks: (date × band, H, W)
    thr = (2 * Fs.flatten(1).std(1) / 4).view(-1, 1, 1)                # similarity threshold per band (4 classes)
    fm, cm = Fs - Fs.mean(0), Cs - Cs.mean(0)                          # centred spectra
    R = (fm * cm).sum(0, keepdim=True) / (fm.norm(dim=0) * cm.norm(dim=0) + 1e-9)  # fine-coarse correlation per pixel
    P = [F.pad(a[None], (r,) * 4, mode='reflect')[0] for a in (Fs, Cs, R, Cp - C1, Cp - C2)]  # padded for window shifts
    wsum = d1 = d2 = n = sx = sy = sxy = sxx = 0                       # running sums over the window
    for oy in range(-r, r + 1, step):  # candidate rows in the window
        for ox in range(-r, r + 1, step):                              # each candidate neighbour offset
            Fj, Cj, Rj, D1, D2 = (a[:, r + oy:r + oy + H, r + ox:r + ox + W] for a in P)  # neighbour values
            sim = ((Fj - Fs).abs() <= thr).all(0, keepdim=True).float()  # similar on both dates and bands?
            w = sim / ((1 - Rj).clamp(min=1e-3) * (1 + (oy ** 2 + ox ** 2) ** .5 / r))  # weight: purer and closer = more
            wsum, d1, d2 = wsum + w, d1 + w * D1, d2 + w * D2           # weighted coarse change since each reference
            x, y = Cj.view(2, 2, H, W), Fj.view(2, 2, H, W)            # (date, band) for the regression
            n, sx, sy = n + 2 * sim, sx + sim * x.sum(0), sy + sim * y.sum(0)  # regression sums: count, Σcoarse, Σfine
            sxy, sxx = sxy + sim * (x * y).sum(0), sxx + sim * (x * x).sum(0)  # regression sums: Σcoarse·fine, Σcoarse²
    var = sxx - sx ** 2 / n                                            # coarse variance among similar pixels
    V = torch.where((n >= 6) & (var > 1e-6), (sxy - sx * sy / n) / var.clamp(min=1e-9), torch.ones_like(var)).clamp(.2, 3)  # conversion coefficient
    box = lambda a: F.avg_pool2d(a[None], win, 1, r, count_include_pad=False)[0].sum(0, keepdim=True)  # window mean
    t1, t2 = 1 / ((box(C1) - box(Cp)).abs() + 1e-6), 1 / ((box(C2) - box(Cp)).abs() + 1e-6)  # trust the less-changed date
    # predict from each reference date, blend by temporal weight
    return ((t1 * (F1 + V * d1 / wsum) + t2 * (F2 + V * d2 / wsum)) / (t1 + t2)).clamp(0, 1).cpu().numpy()

ndvi = lambda x: (x[1] - x[0]) / (x[1] + x[0] + 1e-6)                  # NDVI of an image

def scores(p, t):                                                      # all metrics, on pixels valid in both
    ok = np.isfinite(t[0]) & np.isfinite(p[0]); P, T = p[:, ok], t[:, ok]  # valid pixels; predicted and real values there
    cos = (P * T).sum(0) / (np.linalg.norm(P, axis=0) * np.linalg.norm(T, axis=0) + 1e-12)  # spectral angle cosine
    nP, nT = ndvi(P), ndvi(T)                                          # NDVI from predicted and real
    return dict(RMSE=np.sqrt(((P - T) ** 2).mean()),  # root-mean-square error of reflectance
                # structural similarity, averaged over bands
                SSIM=np.mean([structural_similarity(np.where(ok, p[b], 0), np.where(ok, t[b], 0), data_range=1) for b in range(2)]),
                SAM=np.degrees(np.arccos(np.clip(cos, -1, 1))).mean(),  # spectral angle (degrees)
                # relative global error (scale ratio K)
                ERGAS=100 / K * np.sqrt(((np.sqrt(((P - T) ** 2).mean(1)) / T.mean(1)) ** 2).mean()),
                # NDVI error and R²
                NDVI_RMSE=np.sqrt(((nP - nT) ** 2).mean()), NDVI_R2=1 - ((nP - nT) ** 2).sum() / ((nT - nT.mean()) ** 2).sum())

rows = []                                                              # one row per (date, method)
for d in TEST:  # every held-out date
    a, b = max(r for r in POOL if r < d), min(r for r in POOL if r > d)  # ESTARFM's reference dates around d
    # each method's image for this date
    preds = {'Fusion + blend (final GeoTIFF)': read_pred(d), 'Fusion model': model_only[d], 'Reuse F1': s2[nearest(d, POOL)],
             # ...including the two baselines
             'Time interpolation': time_guess(d), 'ESTARFM': estarfm(filled(a), modis10(a), filled(b), modis10(b), modis10(d))}
    rows += [dict(method=k, **scores(v, s2[d])) for k, v in preds.items()]  # score each method against real S2
# mean scores per method, best NDVI first
display(pd.DataFrame(rows).groupby('method').mean().sort_values('NDVI_RMSE').round(4))

d = TEST[len(TEST) // 2]                                               # one test date to look at
fig, axs = plt.subplots(1, 4, figsize=(16, 4))  # 4 panels
# real vs predicted vs baselines
for ax, (name, img) in zip(axs, {'real S2': s2[d], 'fusion + blend': read_pred(d), 'reuse F1': s2[nearest(d, POOL)], 'MODIS': modis10(d)}.items()):
    # NDVI of a 3 km zoom
    ax.imshow(ndvi(img)[200:500, 200:500], vmin=0, vmax=.9, cmap='RdYlGn'); ax.set_title(f'{name}, NDVI {date(d)}'); ax.axis('off')
plt.show()  # draw

---
## Module 9 · NDVI & crop monitoring (LAST STEP)
**NDVI** from the predicted images (fused days) and real S2 (real days) → **time-series cube** (each day flagged
real / fused / gap) → gaps filled in time and **Savitzky-Golay smoothing** per pixel.

In [ ]:
cube = np.full((N_DAYS, H, W), np.nan, np.float32); source = np.array(['gap'] * N_DAYS, object)  # one layer per day
for d in POOL: cube[d], source[d] = ndvi(s2[d]), 'real'                # NDVI from real S2
for d in DAYS: cube[d], source[d] = ndvi(read_pred(d)), 'fused'        # NDVI from the predicted image
print(pd.Series(source).value_counts().to_dict())  # how many real / fused / gap days

def fill_smooth(c):                                                    # gap-fill in time, then smooth (one row block)
    f, t = c.reshape(N_DAYS, -1), np.arange(N_DAYS)                    # (days, pixels)
    for k in np.flatnonzero(~np.isfinite(f).all(0)):                   # pixels with gaps
        ok = np.isfinite(f[:, k])  # days with a value
        if ok.any(): f[:, k] = np.interp(t, t[ok], f[ok, k])           # linear interpolation over the gaps
    f = np.where(np.isfinite(f), f, np.nan_to_num(np.nanmean(f, 1, keepdims=True)))  # never-seen pixels → day mean
    return savgol_filter(f, 15, 2, axis=0).reshape(c.shape)            # 15-day, 2nd-order smoothing

smooth = np.concatenate([fill_smooth(cube[:, i:i + 50].copy()) for i in range(0, H, 50)], axis=1)  # 50 rows at a time

**Field statistics & alerts**, per field:
* **Stress alert:** in the crop's peak period, the field's own NDVI falls > 0.05 below its best of the last 20 days
  for 5+ days while its crop's median is not falling (so late sowing or a weaker field is not flagged).
* **Patchy:** within-field spread (CV) mid-season > 1.5 × its crop's median. Informational.
* **Phenology:** green-up (half-amplitude going up), peak date and NDVI, senescence (half-amplitude going down).

In [ ]:
fid = field_id.ravel(); inside = fid >= 0                              # field of every pixel; roads excluded
cnt = np.bincount(fid[inside], minlength=N_FIELDS)                     # pixels per field
# mean of NDVI^v per field per day → (fields, days)
per_field = lambda v: np.array([np.bincount(fid[inside], s.ravel()[inside] ** v, N_FIELDS) for s in smooth]).T / np.maximum(cnt, 1)[:, None]
mean_t, sq_t = per_field(1), per_field(2)                              # field mean NDVI and mean square, per day
cv_t = np.sqrt(np.maximum(sq_t - mean_t ** 2, 0)) / np.maximum(mean_t, 1e-3)  # within-field coefficient of variation
drop = lambda x: x - np.array([x[max(0, i - 20):i + 1].max() for i in range(N_DAYS)])  # fall below best of last 20 days

rows = []                                                              # one row per crop field
for c in ('wheat', 'mustard'):  # each crop type
    ids = np.flatnonzero((crop == c) & (cnt > 500)); med = np.median(mean_t[ids], 0)  # fields of this crop; their median
    for f in ids:  # each field of that crop
        x = mean_t[f]; peak = int(x.argmax()); half = (x.min() + x.max()) / 2  # season curve, peak day, half-amplitude
        run = np.convolve((drop(x) < -.05) & (drop(med) > -.02) & (med >= .9 * med.max()), np.ones(5), 'valid') >= 5  # 5-day decline
        # phenology dates...
        rows.append(dict(field=f, crop=c, green_up=date(np.argmax(x >= half)), peak=date(peak), peak_ndvi=round(x.max(), 2),
                         # ...senescence and first stress-alert day...
                         senescence=date(peak + np.argmax(x[peak:] < half)), stress_alert=date(np.argmax(run)) if run.any() else '',
                         patchy=cv_t[f, 90:110].mean() > 1.5 * np.median(cv_t[ids, 90:110].mean(1))))  # ...and patchiness flag
report = pd.DataFrame(rows).set_index('field')                         # full field report
display(report[report.stress_alert != ''])                             # fields with stress alerts
# patchy fields and the injected-stress answer key
print('patchy fields:', list(report.index[report.patchy]), '| injected stress (answer key):', [(int(s['field']), str(date(s['start']))) for s in STRESS])

f = STRESS[0]['field']                                                 # plot one stressed field's season
for src, mk in (('real', 'o'), ('fused', '.')):                        # daily field means, real vs fused days
    dd = [d for d in range(N_DAYS) if source[d] == src]  # days of this source
    plt.plot(dd, [np.nanmean(cube[d][field_id == f]) for d in dd], mk, label=f'{src} days')  # field mean NDVI on those days
# smoothed curve and stress start
plt.plot(mean_t[f], label='smoothed'); plt.axvline(STRESS[0]['start'], ls='--', c='gray', label='stress starts')
plt.xlabel('day of season'); plt.ylabel(f'field {f} mean NDVI'); plt.legend(); plt.show()  # draw